# Step 7 — Benchmarks (Push ~40 Q&A Pairs for Evaluation)

**Mistplay Genie Workshop**

You've tuned the space step by step. Now make quality **measurable**. A benchmark is a set of question + expected-answer pairs Genie's evaluation runs against — turning "is my space good?" into a score you can track and improve (and the input the **Genie Workbench** auto-optimize uses).

We push **~40 questions** spanning the whole difficulty range you built through Steps 2-6:
- basic aggregation, value matching, governed measures, joins/exceptions, and business rules.

Each has a **reference SQL answer**.

> **API vs UI:** benchmarks map to `benchmarks.questions` in `serialized_space` — pushed via REST PATCH. All in code. (Running/scoring the benchmark is then done from the Genie UI or Workbench.)

In [0]:
import os, sys, importlib
nb_dir = os.getcwd()
if nb_dir not in sys.path: sys.path.insert(0, nb_dir)
import workshop_lib.config, workshop_lib.genie_builder, workshop_lib.benchmarks
for m in (workshop_lib.config, workshop_lib.genie_builder, workshop_lib.benchmarks):
    importlib.reload(m)
from workshop_lib.config import WorkshopConfig, setup_widgets
from workshop_lib import genie_builder as gb
from workshop_lib import benchmarks as bench
# Load config saved by notebook 01 (only the 'catalog' widget is needed).
cfg = WorkshopConfig.from_saved(spark, catalog=dbutils.widgets.get("catalog").strip())
client = gb.GenieClient()
space_id = gb.require_space_id(spark, cfg)
print("space:", space_id)

## 1. Preview the benchmark bank

Grouped by category & difficulty. Open `workshop_lib/benchmarks.py` to read/edit them.

In [0]:
bank = bench.build_benchmarks(cfg)
from collections import Counter
print("total questions:", len(bank))
print("by category  :", dict(Counter(b["category"] for b in bank)))
print("by difficulty:", dict(Counter(b["difficulty"] for b in bank)))
print()
for b in bank[:5]:
    print(f"[{b['difficulty']}/{b['category']}] {b['question']}")
print("...")

## 2. Push benchmarks to the space

In [0]:
builder = client.load_builder(space_id)
n = bench.add_all(builder, cfg)
print("added benchmarks:", n)
print(builder.summary())
client.patch_space(space_id, title=cfg.genie_space_name,
                   description="Mistplay Genie Workshop — benchmarked.",
                   warehouse_id=cfg.warehouse_id, serialized_space=builder.to_serialized())
print("Patched.")

## 3. Verify

In [0]:
d = client.load_builder(space_id).to_dict()
qs = d.get("benchmarks",{}).get("questions",[])
print("benchmark questions on space:", len(qs))
print("with reference SQL answer   :", sum(1 for q in qs if q.get("answer")))
print("\nOpen your space and run the benchmark from the UI (or via Genie Workbench):")
print(client.space_url(space_id))

## Workshop complete 🎉

Your Genie space went from a bare table list to a production-grade, **benchmarked** space:

1. Documented gold data → 2. bare space → 3. value matching → 4. metric views (joins + filters) → 5. curated examples → 6. instructions → 7. benchmarks.

**Next (advanced part):** open the space in **Genie Workbench**, run an **IQ scan** for an objective score, and use **auto-optimize** against this benchmark set to close the remaining gaps.